In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import re
import pandas as pd
import random

driver_path = r"F:/chromedriver-win64/chromedriver.exe"

options = Options()
options.page_load_strategy = "eager"
options.add_argument("--headless=new")   # من غير واجهة رسومية = أسرع بكتير
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

service = Service(driver_path)
driver = webdriver.Chrome(service=service, options=options)
driver.set_page_load_timeout(60)

# كل فلتر بيديك مجموعة إعلانات مختلفة - المفتاح الحقيقي لزيادة العدد
locations = [
    "cairo", "giza", "alexandria", "6-of-october-city", "sheikh-zayed",
    "new-cairo", "el-shorouk", "10th-of-ramadan-city", "obour-city"
]

base_url_template = "https://www.dubizzle.com.eg/properties/apartments-duplex-for-sale/{loc}/?page={page}"

all_apartments_data = []
seen_links = set()
TARGET = 3000
MAX_PAGES_PER_LOCATION = 25

for loc in locations:
    print(f"\n=== بندور على منطقة: {loc} ===")
    no_new_streak = 0

    for page_num in range(1, MAX_PAGES_PER_LOCATION + 1):
        url = base_url_template.format(loc=loc, page=page_num)
        try:
            driver.get(url)
            time.sleep(random.uniform(3, 5))   # أسرع من 10 ثواني بسبب headless
        except Exception:
            print(f"  صفحة {page_num}: تعثر اتصال، تخطي")
            continue

        soup = BeautifulSoup(driver.page_source, "html.parser")
        listings = soup.find_all("li")
        listings = [li for li in listings if li.find("a", href=True) and li.get_text(strip=True)]

        page_new = 0
        for item in listings:
            text = item.get_text(separator=" ", strip=True)
            price_match = re.search(r'([\d,]+)\s*ج\.م', text)
            if not price_match:
                continue

            link_tag = item.find("a", href=True)
            link = "https://www.dubizzle.com.eg" + link_tag["href"] if link_tag and link_tag["href"].startswith("/") else None
            if link is None or link in seen_links:
                continue
            seen_links.add(link)

            price = price_match.group(1).replace(",", "")
            rooms_match = re.search(r'(\d+)\s*غرف نوم', text)
            baths_match = re.search(r'(\d+)\s*حمامات', text)
            area_match  = re.search(r'(\d+)\s*م٢', text)
            title_tag = item.find("h2")
            title = title_tag.get_text(strip=True) if title_tag else None

            all_apartments_data.append({
                "title": title,
                "price_egp": price,
                "rooms": rooms_match.group(1) if rooms_match else None,
                "bathrooms": baths_match.group(1) if baths_match else None,
                "area_m2": area_match.group(1) if area_match else None,
                "link": link,
                "search_location": loc
            })
            page_new += 1

        print(f"  صفحة {page_num}: {page_new} جديد | إجمالي كلي: {len(all_apartments_data)}")

        if page_new == 0:
            no_new_streak += 1
            if no_new_streak >= 3:
                print(f"  مفيش جديد من 3 صفحات - خلصنا منطقة {loc}")
                break
        else:
            no_new_streak = 0

        if len(all_apartments_data) >= TARGET:
            break
        time.sleep(random.uniform(2, 4))

    if len(all_apartments_data) >= TARGET:
        print("وصلنا للهدف!")
        break

driver.quit()

df = pd.DataFrame(all_apartments_data)
df = df.drop_duplicates(subset=["link"])
df.to_csv("dubizzle_results_large.csv", index=False, encoding="utf-8-sig")
print(f"\nتم استخراج {len(df)} نقطة بيانات فريدة بنجاح!")
df.head(10)


=== بندور على منطقة: cairo ===
  صفحة 1: تعثر اتصال، تخطي
  صفحة 2: 46 جديد | إجمالي كلي: 46
  صفحة 3: 46 جديد | إجمالي كلي: 92
  صفحة 4: 44 جديد | إجمالي كلي: 136
  صفحة 5: 46 جديد | إجمالي كلي: 182
  صفحة 6: 45 جديد | إجمالي كلي: 227
  صفحة 7: 45 جديد | إجمالي كلي: 272
  صفحة 8: 46 جديد | إجمالي كلي: 318
  صفحة 9: 45 جديد | إجمالي كلي: 363
  صفحة 10: 44 جديد | إجمالي كلي: 407
  صفحة 11: 46 جديد | إجمالي كلي: 453
  صفحة 12: 44 جديد | إجمالي كلي: 497
  صفحة 13: 45 جديد | إجمالي كلي: 542
  صفحة 14: 46 جديد | إجمالي كلي: 588
  صفحة 15: 44 جديد | إجمالي كلي: 632
  صفحة 16: 45 جديد | إجمالي كلي: 677
  صفحة 17: 45 جديد | إجمالي كلي: 722
  صفحة 18: 45 جديد | إجمالي كلي: 767
  صفحة 19: 44 جديد | إجمالي كلي: 811
  صفحة 20: 45 جديد | إجمالي كلي: 856
  صفحة 21: 44 جديد | إجمالي كلي: 900
  صفحة 22: 44 جديد | إجمالي كلي: 944
  صفحة 23: 45 جديد | إجمالي كلي: 989
  صفحة 24: 44 جديد | إجمالي كلي: 1033
  صفحة 25: 45 جديد | إجمالي كلي: 1078

=== بندور على منطقة: giza ===
  صفحة 1: 0 جديد | إجمالي كلي:

,title,price_egp,rooms,bathrooms,area_m2,link,search_location
0,شقتك شقتين امتلك شقة مع استوديو مستقل في R8,6389100,4,2,229,https://www.dubizzle.com.eg/ad/%D8%B4%D9%82%D8...,cairo
1,شقه للبيع { تسليم فوري ومتشطبه } في كمبوند الم...,3902792,3,2,134,https://www.dubizzle.com.eg/ad/%D8%B4%D9%82%D9...,cairo
2,استاند الون شقه للبيع ( استلام فوري ) فيو مفتو...,18712972,3,3,204,https://www.dubizzle.com.eg/ad/%D8%A7%D8%B3%D8...,cairo
3,شقه متشطبه من waterway ڤيو lakeبخصم 18% من سعر...,17515000,3,3,160,https://www.dubizzle.com.eg/ad/%D8%B4%D9%82%D9...,cairo
4,Apartment for sale,11000000,2,2,300,https://www.dubizzle.com.eg/ad/apartment-for-s...,cairo
5,فرصة استثنائية للشراء شقة جاهزة للأستلام 213 م...,13300000,3,3,212,https://www.dubizzle.com.eg/ad/%D9%81%D8%B1%D8...,cairo
6,فرصة لا تُعوَّض الآن شقة 200 م للبيع استلام فو...,9500000,3,2,200,https://www.dubizzle.com.eg/ad/%D9%81%D8%B1%D8...,cairo
7,فرصة متتعوضش! شقة 162م في برايم لوكيشن أمام ال...,6797000,3,3,162,https://www.dubizzle.com.eg/ad/%D9%81%D8%B1%D8...,cairo
8,"بقسط شهري يبدأ من 16,000 جنيه فقط يمكنك امتلاك...",4300000,3,2,155,https://www.dubizzle.com.eg/ad/%D8%A8%D9%82%D8...,cairo
9,شقه للبيع تشطيب الترا مودرن فيو علي البرج الاي...,3350000,3,2,134,https://www.dubizzle.com.eg/ad/%D8%B4%D9%82%D9...,cairo
